In [ ]:
# import packages
import numpy as np 
import pandas as pd
import re

In [2]:
# import data 
df = pd.read_csv("../data/hot100_history.csv", low_memory=False)

In [3]:
# add year column
df["chart_date"] = pd.to_datetime(df["chart_date"])
df["year"] = df["chart_date"].dt.year

In [26]:
# null values 
df.isna().sum()

chart_date        0
rank              0
title             0
artist            0
weeks_on_chart    0
peak_position     0
last_position     0
is_new            0
year              0
dtype: int64

In [23]:
# get rid of html tags 
def strip_html(s):
    return re.sub(r'<[^>]+>', '', s).strip()

df['artist'] = df['artist'].apply(strip_html)

In [ ]:
# expand collabs and features into an artist list column 

# list of artists to not separate
BAND_BLOCKLIST = [
    'X Ambassadors',
    'Lil Nas X',
    'Artists Of Then, Now & Forever',
    'Dan + Shay',
    'Florence + The Machine',
    'Tyler, The Creator',
    'Fitz And The Tantrums',
    'Mitchell Ayers And His Orchestra',
    'Mitchell Ayres And His Orchestra',
    'Tones And I',
    'John Scott Trotter And His Orchestra',
    'John Scott Trotter & His Orchestra', 
    'Bobby "Boris" Pickett And The Crypt-Kickers',
    'Henri Rene And His Orchestra',
    'Henri Rene and His Orchestra',
    'The War And Treaty',
    'She & Him',
    'Richy Mitch And The Coal Miners',
    "4*TOWN (From Disney And Pixar's Turning Red)",
    'HUNTR/X',
    'The Orchestra & Chorus Of Gordon Jenkins'
]

# function to split artists 
def split_artists(artist_str):
    # replace blocklisted names with placeholder tokens before splitting
    protected = artist_str
    replacements = {}
    for i, band in enumerate(BAND_BLOCKLIST):
        pattern = re.compile(re.escape(band), re.IGNORECASE)
        if pattern.search(protected):
            token = f"__BAND_{i}__"
            replacements[token] = band
            protected = pattern.sub(token, protected)
    
    # remove parenthesis 
    protected = re.sub(r'[()]', '', protected).strip()

    # now split
    SEP = r'\s*(?:Featuring|Present\.\.\.|Duet With|Presents|feat\.|Feat\.|With|[&,/]|\s+[Aa]nd\s+|\s+[Oo]r\s+|\s+[Xx]\s+|\s+\+\s+|\:\s+)\s*'
    parts = re.split(SEP, protected)
    parts = [p.strip() for p in parts if p.strip()]

    # restore placeholders to original band names
    result = []
    for part in parts:
        for token, original in replacements.items():
            part = part.replace(token, original)
        result.append(part)

    return result if result else [artist_str.strip()]

In [25]:
# add artist list 
df['artist_list'] = df['artist'].apply(split_artists)

In [26]:
# save cleaned data 
df.to_csv("../data/clean_hot100_history.csv", index=False)